# Phase 1: 프로젝트 초기화 및 환경 설정

## 🎯 이 노트북에서 배울 것
- 현대적인 파이썬 패키지 관리자 **`uv`**의 기초 사용법을 익힙니다.
- 프로그램의 비빌번호인 **환경변수(`.env`)**를 왜 쓰는지, 어떻게 관리하는지 배웁니다.
- 프로젝트의 설정을 한곳에 모아 관리하는 **`Settings` 클래스** 구조를 이해합니다.

## 📋 사전 준비
- `uv`가 설치되어 있어야 합니다 (README.md 참고).
- `.env` 파일에 API 키가 설정되어 있어야 실습 결과를 확인할 수 있습니다.

## 📖 학습 순서
1. **이론 학습** - 패키지 관리와 환경변수의 개념 이해
2. **가이드 실습** - 파이썬으로 환경변수를 직접 다뤄보기
3. **프로젝트 코드 분석** - 실제 `settings.py`가 에러를 보여주는 방식 분석
4. **강사와 함께하는 챌린지** - 나만의 설정 로더 만들어보기

---

## 📚 1. 패키지 관리자 `uv`란?

### 이게 뭔가요?
`uv`는 파이썬에서 필요한 **재료(라이브러리)**를 아주 빠르게 가져다주는 **'스마트 장보기 도구'**입니다.
- 요리(프로그래밍)를 할 때 필요한 밀가루, 소금(패키지)을 대신 사오고,
- 유통기한(버전)이 지난 재료는 없는지 관리해줍니다.

### 왜 필요한가요?
기존 도구(pip)보다 수십 배 빠르고, **'가상환경'**이라는 독립된 주방을 자동으로 만들어주어 요리가 섞이지 않게 도와줍니다.

### 실행해서 확인해봅시다!
아래 코드는 현재 주방(환경)에 어떤 재료들이 설치되어 있는지 확인하는 코드입니다.

In [ ]:
# ============================================
# 🔍 설치된 라이브러리 목록 확인하기
# ============================================
# 주피터 노트북에서 '!'를 붙이면 터미널 명령어를 실행합니다.

!uv pip list

# 💡 실행 결과를 보세요!
# - streamlit, google-genai, pandas 등 우리가 사용할 재료들이 보이나요?

---

## 📚 2. 환경변수(`.env`)란?

### 이게 뭔가요?
`.env` 파일은 프로그램의 **'비밀 수첩'**입니다.
- 금고 비밀번호(API 키), 서버 주소처럼 **코드에 직접 쓰면 위험한 정보**를 따로 적어둡니다.

### 왜 필요한가요?
인터넷(GitHub 등)에 코드를 올릴 때, 비밀번호가 적힌 수첩은 집에 두고 **요리법(코드)만 공유**하기 위해서입니다.

### 실행해서 확인해봅시다!
파이썬에서 수첩(`.env`)의 내용을 읽어오는 일반적인 방법입니다.

In [ ]:
# ============================================
# 🔍 환경변수 읽기 기본 동작
# ============================================
import os
from dotenv import load_dotenv

# 1. .env 파일의 내용을 파이썬으로 가져옵니다.
load_dotenv()

# 2. os.getenv 함수를 써서 특정 이름을 가진 정보를 꺼냅니다.
model_name = os.getenv("GEMINI_MODEL", "기본값-flash") # 정보가 없으면 기본값을 씁니다.

print(f"사용할 AI 모델: {model_name}")
print(f"데이터 저장 경로: {os.getenv('CSV_PATH')}")

# 💡 팁: .env 파일이 없거나 내용이 비어있으면 None이 나옵니다.

---

## ✏️ 실습: 환경변수 다루기 가이드

배운 내용을 직접 코딩해봅시다!

**목표**: 특정 환경변수가 없을 때 친절하게 알려주는 코드 만들기

In [ ]:
# ============================================
# ✏️ 실습: 환경변수 체크 함수 만들기
# ============================================
import os
from dotenv import load_dotenv

load_dotenv()

# TODO 1: os.getenv를 사용해서 'TAVILY_API_KEY'를 가져와 api_key 변수에 저장하세요.
api_key = ""

# >>> 정답 (학습 후 주석 해제하여 확인) <<<
# api_key = os.getenv("TAVILY_API_KEY")

# TODO 2: 만약 api_key가 비어있다면(None이라면) "키가 없어요!"라고 출력하세요.

# >>> 정답 <<<
# if not api_key:
#     print("키가 없어요!")

print(f"가져온 키: {api_key[:5]}...") # 보안을 위해 앞 5자리만 출력

---

## 🔎 실제 프로젝트 코드 분석

우리 프로젝트의 `config/settings.py`는 왜 이렇게 만들어졌을까요?

### 핵심 로직 분석 (`_get_required_env` 함수)
이 함수는 단순히 환경변수를 읽는 게 아니라, **없을 때 해결법(URL)까지 같이 제공**하는 아주 친절한 함수입니다.

In [ ]:
# ============================================
# 🔎 initial_version/config/settings.py 분석
# ============================================

def _get_required_env(key, error_msg):
    import os
    value = os.getenv(key) # 수첩에서 값을 찾고
    if not value:          # 없으면?
        # 그냥 에러가 아니라 '친절한 안내문'을 만듭니다.
        guide = f"❌ {key}가 누락되었습니다.\n설정 방법: 발급 링크 {error_msg}"
        raise ValueError(guide) # 프로그램을 멈추고 안내문을 보여줍니다.
    return value

# 💡 왜 이렇게 하나요?
# - 개발을 아예 모르는 사람이 앱을 켰을 때, 
# - 모르는 에러 코드 대신 '어디 가서 키를 받아오세요'라는 안내를 보기 위해서입니다.

---

## 🏆 강사와 함께하는 챌린지

⚠️ **주의**: 여기에는 정답이 없습니다. 직접 시도해보고 강사님과 함께 풀어보세요!

**챌린지 목표**: 사용자 이름을 환경변수에서 읽어오되, 없으면 "손님"이라고 출력하고 이름을 적는 수법을 안내하는 클래스 만들기

In [ ]:
# ============================================
# 🏆 챌린지: 나만의 환경설정 로더 만들기
# ============================================

class MyConfig:
    def __init__(self):
        import os
        # TODO 1: 'USER_NAME' 환경변수를 가져오세요. 없으면 "손님"을 기본값으로 하세요.
        self.user_name = "TODO"
        
        # TODO 2: 'APP_THEME' 환경변수를 확인하여 없으면 
        # ".env 파일에 APP_THEME=dark 처럼 적어주세요" 라는 안내문을 출력하세요.
        self.theme = "TODO"

# 실행 코드
config = MyConfig()
print(f"인사: 안녕하세요 {config.user_name}님!")

## 📋 정리

### 이번 Phase에서 배운 것
| 개념 | 한 줄 요약 |
|------|-----------|
| `uv` | 속도가 아주 빠른 파이썬 스마트 장보기 도구 |
| `.env` | 소중한 비밀번호(API 키)를 따로 적어둔 비밀 수첩 |
| `ValueError` | 잘못된 상황일 때 친절한 안내를 띄우며 멈추는 방법 |

### 다음 Phase에서는?
- 우리가 다룰 데이터(뉴스, 결과)를 파이썬의 언어로 정의하는 **'도메인 모델'**을 만듭니다.
- 데이터를 담는 예쁜 그릇인 `dataclass`를 배우게 될 거예요!

## ✅ 학습 체크리스트
- [ ] `!uv pip list`를 실행하여 어떤 패키지가 있는지 확인했다.
- [ ] `.env` 파일을 읽어오는 `os.getenv`의 역할을 이해했다.
- [ ] 필수값이 없을 때 `raise`로 에러를 띄우는 이유를 이해했다.